# DS4420 Project - Collaborative Filtering Model

Authors: Gavin Bond, Benjamin Kataoka, Preetish Paul

## User-User Collaborative Filtering

This notebook implements our user-user collaborative filtering model to predict how much developers trust AI generated output (AIAcc). We load the cleaned combined dataset exported from preprocessing.ipynb and build from there.

In [24]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score

In [25]:
# Load the cleaned combined dataset from preprocessing.ipynb
df = pd.read_csv("data/combined_survey.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print("Year breakdown:")
print(df["year"].value_counts())

Shape: (55118, 14)
Columns: ['YearsCode', 'WorkExp', 'Age', 'DevType', 'EdLevel', 'OrgSize', 'AISelect', 'AISent', 'AIThreat', 'RemoteWork', 'AIComplex', 'ICorPM', 'AIAcc', 'year']

Year breakdown:
year
2024    29379
2025    25739
Name: count, dtype: int64


### Target Encoding

We ordinally encode AIAcc on a 0 to 4 scale rather than binarizing like the NN does. CF predicts a continuous score from similar users so keeping the full range gives us richer signal to work with.

In [26]:
trust_map = {
    "Highly trust": 4,
    "Somewhat trust": 3,
    "Neither trust nor distrust": 2,
    "Somewhat distrust": 1,
    "Highly distrust": 0
}

df["AIAcc_encoded"] = df["AIAcc"].map(trust_map)

print("AIAcc distribution:")
print(df["AIAcc_encoded"].value_counts().sort_index())

AIAcc distribution:
AIAcc_encoded
0     7490
1    13536
2    13582
3    19144
4     1366
Name: count, dtype: int64


### Feature Cleaning

We convert the numeric columns to numbers since they come in as text, fill missing values, and clean up the categorical columns. We also borrow the reduce_categories approach from the NN notebook to drop any category that appears fewer than 100 times. This prevents the one-hot encoding from creating a ton of sparse columns from rare DevType or EdLevel values.

In [27]:
numeric_cols = ["YearsCode", "WorkExp"]
categorical_cols = [
    "Age", "DevType", "EdLevel", "OrgSize",
    "AISelect", "AISent", "AIThreat", "RemoteWork",
    "AIComplex", "ICorPM"
]

# Convert numeric columns and fill with median
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Fill categorical NaN
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown").astype(str)

# Collapse rare categories (fewer than 100 occurrences) into "Other"
def reduce_categories(series, min_count=100):
    counts = series.value_counts()
    keep = counts[counts >= min_count].index
    return series.where(series.isin(keep), "Other")

for col in categorical_cols:
    df[col] = reduce_categories(df[col])

print("Remaining NAs:", df[numeric_cols + categorical_cols].isna().sum().sum())
print("Dataset shape:", df.shape)

Remaining NAs: 0
Dataset shape: (55118, 15)
